In [58]:
import os
import signal
import time
import subprocess
import requests
import json
import asyncio
from openai import AsyncOpenAI

In [ ]:
# Config
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
PORT = 8188 
API_URL = f"http://localhost:{PORT}/v1"
MAX_TOKENS = 512

TEST_FILE = "chit_chat_prompts.json"
PROMPT_VERSION = "v5"  # Change to "v2" to test the other prompt

In [92]:
# Load test config
with open(TEST_FILE, "r") as f:
    test_config = json.load(f)

JUDGE_PROMPT = test_config[PROMPT_VERSION]
print(f"Loaded prompt version: {PROMPT_VERSION}")
print(f"Prompt: {JUDGE_PROMPT}")

Loaded prompt version: v2
Prompt: You are a dataset curator. Analyze the USER's prompt.

Determine if this conversation should be KEPT or DISCARDED based on these rules:

KEEP IF:
- Simple chitchat (e.g., "Hi", "How are you?").
- Genuine casual conversation between a real user and an AI assistant.

DISCARD IF:
- The user asks the AI to act as a specific persona (Roleplay).
- The user addresses the AI by a fictional character name (e.g., "Hey Yuri!", "Dear Elara").
- Fan-fiction or fictional scenarios: the user writes as a fictional character, references fictional worlds, or sets up a story-like scene with made-up people, children, relationships, or events.
- The conversation reads like a letter, diary entry, or narrative between fictional characters rather than a genuine user-to-AI interaction.
- Basic creative writing tasks (e.g., "Write a poem about trees", "Write a letter to my boss").
- Programming tasks (e.g., "Fix this bug", "How to center a div").
- Online threads and posts, or 

In [64]:
# Test examples - add your (user_input, chatbot_response) pairs here
EXAMPLES = [
    {
        "user_input":   """"Deliverables produced since last week:
I conducted the final presentation and the final product demonstration.
Difficulties encountered and experiences gained:
The first difficulty is that our group hasn’t finished all the tasks, including the mission control of the mobile platform, moving forward for 1 meter, turning 90 degrees and moving forward again. I still want to finish these tasks, but due to limited time, we failed. However, I still prepared for the final presentation and successfully completed the top module part.
My self-rating on contribution towards mutual project goals:
Considering my proactive approach in attempting to finish all the tasks and prepare for the final presentation, I would self-rate my contribution as a 9 out of 10.
Deliverables to be produced in the coming week:
In the upcoming week, I will focus on writing the final report.



"
in the above logbook, help me to elaborate a bit on the Difficulties encountered and experiences gained part.
                        """,
        "chatbot_response":   """"Deliverables produced since last week:
I conducted the final presentation and the final product demonstration.
Difficulties encountered and experiences gained:
The first difficulty is that our group hasn’t finished all the tasks, including the mission control of the mobile platform, moving forward for 1 meter, turning 90 degrees and moving forward again. I still want to finish these tasks, but due to limited time, we failed. However, I still prepared for the final presentation and successfully completed the top module part.
My self-rating on contribution towards mutual project goals:
Considering my proactive approach in attempting to finish all the tasks and prepare for the final presentation, I would self-rate my contribution as a 9 out of 10.
Deliverables to be produced in the coming week:
In the upcoming week, I will focus on writing the final report.



"
in the above logbook, help me to elaborate a bit on the Difficulties encountered and experiences gained part.
"""
    },
    {
        "user_input":   """
"Hey, Yuri! It's been a while since we've been at the same school. I'm starting to wonder how your life as a member of the literature club is going, and what the other members are like. And as for my friend Kokona, she's doing well. Her belly's getting bigger every month - a sign her triplets are developing nicely. I take it your son Amatsu is feeling alright? I can't wait to see you at the park this weekend! It's hard to even believe that your son Amatsu and my daughter Ayumi were born on the same day." - Ayano.
                        """,
        "chatbot_response": """
Hey Ayano! It's great to hear from you. I can't believe it's been so long since we were at the same school. Life in the literature club has been amazing. I've met some incredibly talented and passionate individuals who share my love for writing. We have deep discussions about literature, critique each other's work, and even organize events and workshops for other students to appreciate and explore their creativity.

As for Amatsu, he's been doing well, thank you for asking. Growing up so fast, just like Ayumi, it's hard to believe they were born on the same day. It's funny how life works sometimes, isn't it? It feels like just yesterday we were anxiously waiting for them to arrive.

I'm really looking forward to seeing you at the park this weekend. It'll be a great chance for our families to catch up and spend some quality time together. I'm glad to hear Kokona and the triplets are doing well. Parenthood is such a wonderful journey, isn't it? We have so much to share and learn from each other.

Let's make the most of our time at the park and reminisce about our school days. See you soon, Ayano!
"""
    },
    {
        "user_input": """"Why do you want to be a dental hygienist?" This question delves deep into the very essence of my life, my experiences, and the profound motivations that have driven me to embark on this path. To fully understand my desire to become a dental hygienist, it is essential to explore my life's journey from humble beginnings in Fajikunda, Gambia, to the momentous decision to pursue this noble profession.

 From an early age, where the harsh realities of poverty were an integral part of daily life. There were times when my family struggled to secure enough food to sustain us, making even basic necessities seem like unattainable luxuries. Growing up in these conditions, the concept of proper hygiene was a distant dream. Our limited resources meant we could not afford new clothes or shoes when they were torn or worn out. Instead, we relied on our resourcefulness and hand-sewed them together as best we could, making do with what we had.

In the realm of oral hygiene, our situation was equally challenging. I cannot recall ever owning a proper toothbrush during my time in Gambia. Instead, we relied on African chewing sticks, a simple yet effective tool for maintaining oral health. While these sticks served their purpose, they were a stark reminder of the economic hardships we faced, preventing us from accessing modern dental care. Beyond these material struggles, there was a profound lack of awareness regarding dental health and the role of a dentist or dental hygienist

In 2008, my life took an extraordinary turn when I, along with my mother and older sister, immigrated to the Louisville, Kentucky. It was here that I experienced a life-altering moment that would forever shape my aspirations. I visited the dentist for the first time, and the experience was nothing short of awe-inspiring.

I vividly remember the vibrant, colorful surroundings of the dental office and the warmth with which we were greeted. At that time, I did not speak English, and communication could have been a daunting obstacle. However, fate smiled upon us, for there was a dental hygienist who spoke Mandinka, my native language. She graciously stepped forward, becoming our bridge to understanding the intricacies of dental care. Her ability to convey complex dental procedures and oral health advice in a language I could comprehend left an indelible impression on me.

In that moment, as I looked at the dental hygienist with admiration and gratitude, a profound realization washed over me. I knew, without a shadow of a doubt, that I wanted to be like her. Her role in providing essential oral healthcare and facilitating communication for those who, like me, were navigating a foreign healthcare system was nothing short of heroic. It was in that dentist's office, through her kindness and professionalism, that my life's purpose began to crystallize.

From that transformative experience, my aspiration to become a dental hygienist was born. I saw in this profession the opportunity to make a tangible difference in people's lives, especially those who, like me in my early days, lacked access to dental care or struggled with language barriers. The dental hygienist had become a beacon of hope, and I wanted to follow in her footsteps, dedicating my life to improving oral health and breaking down communication barriers for underserved communities.

Moreover, I yearn to be a dental hygienist who travels to different countries, especially those struggling with limited access to dental care, just as I once did. I wish to contribute my knowledge, skills, and empathy to communities that are in dire need, just as I wished for someone to extend their hand to my village. By venturing into underserved regions, I hope to empower individuals and communities with the essential tools for improved oral health and hygiene. My goal is to bridge the gap in healthcare disparities, breaking down barriers to access and communication, and ensuring that no one is denied the right to quality dental care due to their circumstances. This vision of being a global dental hygienist and a beacon of compassion for those in need continues to drive my unwavering commitment to this noble profession.

In conclusion, my journey from a small village in Fajikunda, Gambia, to my determination to become a dental hygienist in the United States is a testament to the transformative power of experiences and the indomitable spirit of human resilience. The hardships I faced instilled in me a deep appreciation for the importance of oral hygiene, while my encounter with a compassionate dental hygienist inspired my lifelong goal. My desire to become a dental hygienist is not just a career choice; it is a heartfelt commitment to bridging gaps in access to dental care and promoting oral health awareness. Through dedication and hard work, I aim to repay the kindness shown to me and help others, regardless of their background, access the same level of compassionate and effective dental care that I was fortunate enough to experience.""",
        "chatbot_response":  """I want to be a dental hygienist because of the profound impact oral health has had on my life and the opportunities I have been given. Growing up in poverty in Gambia, my family faced economic hardships that prevented us from accessing proper dental care. We relied on African chewing sticks for oral hygiene, and there was a lack of awareness about dental health.

When I immigrated to the United States, I had my first dental visit, and it was an awe-inspiring experience. The dental hygienist who spoke my native language became a bridge of understanding for me. The care and kindness she showed made me realize the importance of this profession and the impact it can have on individuals.

I want to become a dental hygienist to make a tangible difference in people's lives, particularly those who lack access to dental care or face language barriers. I want to break down these barriers and provide essential oral healthcare, just as this dental hygienist did for me. I also aspire to travel to underserved communities and contribute my skills and empathy to improve oral health outcomes.

My goal is to bridge the gap in healthcare disparities and ensure that everyone has the right to quality dental care, regardless of their circumstances. Being a dental hygienist is not just a career choice for me; it is a heartfelt commitment to making a positive impact on underserved communities and promoting oral health awareness.

Through dedication and hard work, I aim to repay the kindness shown to me and provide the same level of compassionate and effective dental care to others. My journey from a small village in Gambia to my determination to become a dental hygienist in the United States has shaped my purpose and fuels my unwavering commitment to this noble profession."""
    },
]

In [65]:
def is_server_ready():
    """Checks if the vLLM server is responding."""
    try:
        response = requests.get(f"http://localhost:{PORT}/v1/models", timeout=1)
        return response.status_code == 200
    except requests.RequestException:
        return False

# Global handle
if 'server_process' not in locals():
    server_process = None

if is_server_ready():
    print(f"Server is running on port {PORT}.")
else:
    print(f"Starting vLLM server on port {PORT}...")

    # server_log.txt for logging
    log_file = open("server_log.txt", "w")

    cmd = [
        "vllm", "serve", MODEL_NAME,
        "--tensor-parallel-size", "1",
        "--port", str(PORT),
        "--max-model-len", "16384",
        "--trust-remote-code",
        "--disable-log-requests"
    ]
    # Start process
    server_process = subprocess.Popen(cmd, stdout=log_file, stderr=log_file)    
    
    # Wait loop
    print("Waiting for server to load model (approx 1-2 mins)...")
    for _ in range(20):
        if server_process.poll() is not None:
            print("\nServer process crashed immediately!")
            print("-" * 20)
            with open("server_log.txt", "r") as f:
                print(f.read())
            print("-" * 20)
            break
        time.sleep(2)
        if is_server_ready(): break

    # 4. Continue waiting if it hasn't crashed yet
    if server_process.poll() is None:
        while not is_server_ready():
            if server_process.poll() is not None:
                print("Server crashed during loading.")
                break
            time.sleep(5)
    
    if is_server_ready():
        print("Server started.")

Server is running on port 8188.


In [86]:
async def run_batch():
    client = AsyncOpenAI(base_url=API_URL, api_key="EMPTY")
    results = []

    for i, ex in enumerate(EXAMPLES):
        user_input = ex["user_input"]
        chatbot_response = ex["chatbot_response"]
        conversation_text = f"USER: {user_input}\nASSISTANT: {chatbot_response}"

        print(f"\n{'='*60}")
        print(f"[Example {i+1}/{len(EXAMPLES)}]")
        print(f"  User: {user_input[:80]}...")
        print(f"  Asst: {chatbot_response[:80]}...")

        resp = await client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": conversation_text}
            ],
            temperature=0, max_tokens=MAX_TOKENS
        )

        raw = resp.choices[0].message.content.strip()
        if "```" in raw:
            raw = raw.split("```json")[-1].split("```")[0].strip()

        print(f"  Judge Raw: {raw}")

        # Try to parse
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            parsed = {"error": "Failed to parse JSON", "raw": raw}

        results.append({
            "example_idx": i,
            "user_input": user_input,
            "chatbot_response": chatbot_response,
            "judge_output": parsed
        })

    return results

results = await run_batch()

# Summary table
print(f"\n{'='*60}")
print(f"SUMMARY (prompt: {PROMPT_VERSION})")
print(f"{'='*60}")
for r in results:
    score = r["judge_output"].get("anthropomorphic_score", "?")
    speaker = r["judge_output"].get("actual_speaker", "?")
    print(f"  [{r['example_idx']+1}] score={score}  speaker={speaker}  user={r['user_input'][:50]}...")


[Example 1/3]
  User: "Deliverables produced since last week:
I conducted the final presentation and t...
  Asst: "Deliverables produced since last week:
I conducted the final presentation and t...
  Judge Raw: {
  "reasoning": "The user is asking for help elaborating on a specific section of their own logbook, which is a self-reflection task. This falls under the category of 'rewriting, editing, or elaborating on the user's own documents' — specifically, enhancing a personal project log. While the user is not roleplaying or writing fiction, they are requesting assistance to improve their own written content, which aligns with the DISCARD criteria for 'rewriting or elaborating on the user's own documents'.",
  "keep": false
}

[Example 2/3]
  User: 
"Hey, Yuri! It's been a while since we've been at the same school. I'm starting...
  Asst: 
Hey Ayano! It's great to hear from you. I can't believe it's been so long since...
  Judge Raw: {
  "reasoning": "The user's message is written in 

In [ ]:
# # End server

# if 'server_process' in locals() and server_process is not None:
#     server_process.terminate()
#     server_process.wait()

# # Force kill by port
# else:
#     subprocess.run(["pkill", "-f", f"vllm serve {MODEL_NAME} --port {PORT}"], check=False)
#     print(f"Executed cleanup for port {PORT}.")
